In [51]:
from prefect import flow, task
import psycopg2
import os
from dotenv import load_dotenv

load_dotenv()

True

In [52]:
# DB CONNECTION
def get_connection():
    return psycopg2.connect(
        host=os.getenv("DB_HOST"),
        database=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        port=os.getenv("DB_PORT"),
        sslmode="require"
    )

def run_query(query):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute(query)
    conn.commit()
    cur.close()
    conn.close()

In [53]:
# TASK 1: CREATE CLEAN TABLE
# -------------------------------
@task
def create_weather_clean():
    query = """
    DROP TABLE IF EXISTS weather_clean;

    CREATE TABLE weather_clean AS
    SELECT
        location_id,
        timestamp,
        temperature,
        humidity,
        wind_speed,
        precipitation::DOUBLE PRECISION AS precipitation,
        wind_direction,
        cloud_cover,
        dew_point,
        apparent_temperature,
        pressure_msl,
        weather_code
    FROM weather_data
    WHERE temperature IS NOT NULL;
    """
    run_query(query)
    print("weather_clean created")

In [54]:
# TASK 2: FEATURE ENGINEERING
# -------------------------------
@task
def create_weather_features():
    query = """
    DROP VIEW IF EXISTS city_rankings;
    DROP TABLE IF EXISTS weather_features;

    CREATE TABLE weather_features AS
    SELECT
        location_id,
        timestamp,
        temperature,
        humidity,
        wind_speed,
        precipitation,
        EXTRACT(HOUR FROM timestamp) AS hour,
        EXTRACT(DAY FROM timestamp) AS day,
        EXTRACT(MONTH FROM timestamp) AS month,
        CASE
            WHEN EXTRACT(MONTH FROM timestamp) IN (12,1,2) THEN 'Winter'
            WHEN EXTRACT(MONTH FROM timestamp) IN (3,4,5) THEN 'Spring'
            WHEN EXTRACT(MONTH FROM timestamp) IN (6,7,8) THEN 'Summer'
            ELSE 'Autumn'
        END AS season
    FROM weather_clean;
    """
    run_query(query)
    print("weather_features created")

In [55]:
# -------------------------------
# TASK 3: DAILY SUMMARY
# -------------------------------
@task
def create_daily_summary():
    query = """
    DROP TABLE IF EXISTS weather_daily_summary;

    CREATE TABLE weather_daily_summary AS
    SELECT
        location_id,
        DATE(timestamp) AS date,
        AVG(temperature) AS avg_temperature,
        AVG(humidity) AS avg_humidity,
        SUM(precipitation) AS total_precipitation,
        AVG(wind_speed) AS avg_wind_speed
    FROM weather_features
    GROUP BY location_id, DATE(timestamp);
    """
    run_query(query)
    print("weather_daily_summary created")

In [56]:
# CITY RANKINGS
# -------------------------------
@task
def create_city_rankings():
    query = """
    DROP VIEW IF EXISTS city_rankings;

    CREATE VIEW city_rankings AS
    SELECT
        location_id,
        AVG(temperature) AS avg_temp,
        RANK() OVER (ORDER BY AVG(temperature) DESC) AS temp_rank
    FROM weather_features
    GROUP BY location_id;
    """
    run_query(query)
    print("city_rankings view created")

In [57]:
# TASK 5: ANOMALY DETECTION
# -------------------------------
@task
def create_anomalies():
    query = """
    DROP TABLE IF EXISTS weather_anomalies;

    CREATE TABLE weather_anomalies AS
    SELECT *
    FROM (
        SELECT
            location_id,
            timestamp,
            temperature,
            AVG(temperature) OVER (PARTITION BY location_id) AS avg_temp,
            temperature - AVG(temperature) OVER (PARTITION BY location_id) AS deviation
        FROM weather_features
    ) sub
    WHERE ABS(deviation) > 5;
    """
    run_query(query)
    print("weather_anomalies created")

In [58]:
# MAIN FLOW
# -------------------------------
@flow(name="weather-etl-pipeline")
def weather_pipeline():
    clean = create_weather_clean()
    features = create_weather_features(wait_for=[clean])
    summary = create_daily_summary(wait_for=[features])
    rankings = create_city_rankings(wait_for=[features])
    anomalies = create_anomalies(wait_for=[features])

In [59]:
# RUN
# -------------------------------
if __name__ == "__main__":
    weather_pipeline()

20:22:19.201 | INFO    | Flow run 'gleaming-cicada' - Beginning flow run 'gleaming-cicada' for flow 'weather-etl-pipeline'

weather_clean created


20:22:29.956 | INFO    | Task run 'create_weather_clean-40c' - Finished in state Completed()

weather_features created


20:22:35.751 | INFO    | Task run 'create_weather_features-523' - Finished in state Completed()

...

city_rankings view created


20:22:44.258 | INFO    | Task run 'create_city_rankings-400' - Finished in state Completed()

weather_anomalies created


20:22:49.072 | INFO    | Task run 'create_anomalies-577' - Finished in state Completed()

20:22:49.235 | INFO    | Flow run 'gleaming-cicada' - Finished in state Completed()